# Footshocks in sessions converted from CSV

The CSV behavior reader never read the shock column `H`, so footshock probes converted from CSV exports have no `footshock_times`. In DANDI:000971 these are the 13 DLS Excitatory probes whose per-subject copy came from a CSV. The same runs were also published a second time from the date-organized MED-PC files (treatment "Unknown", removed by the duplicate fix in #43), and those copies have the shocks. This notebook reconverts the 13 CSV sessions and compares their footshocks with both published copies.

In [1]:
from pathlib import Path
import shutil

import h5py
import numpy as np
import pandas as pd

from lerner_lab_to_nwb.seiler_2024 import Seiler2024CSVBehaviorInterface
from lerner_lab_to_nwb.seiler_2024.seiler_2024_convert_dataset import opto_to_nwb
from lerner_lab_to_nwb.seiler_2024.seiler_2024_convert_session import session_to_nwb

data_dir_path = Path("/Volumes/T7/CatalystNeuro/Lerner/_Raw_Data_for_future_reference")
published_dir_path = Path("/Volumes/T7/CatalystNeuro/Lerner/000971")
output_dir_path = Path("/Volumes/T7/CatalystNeuro/Lerner/fix_audit/csv_footshock")
if output_dir_path.exists():
    shutil.rmtree(output_dir_path)
output_dir_path.mkdir(parents=True)

In [2]:
footshock_msns = {"Footshock Degradation Left", "Footshock Degradation right"}
csv_footshock_kwargs = []
for session_to_nwb_kwargs in opto_to_nwb(
    data_dir_path=data_dir_path, output_dir_path=output_dir_path, start_variable="Start Date", verbose=False
):
    behavior_file_path = session_to_nwb_kwargs["behavior_file_path"]
    if behavior_file_path.suffix.lower() != ".csv":
        continue
    behavior_metadata = Seiler2024CSVBehaviorInterface(file_path=str(behavior_file_path)).get_metadata()["Behavior"]
    if behavior_metadata["MSN"] in footshock_msns:
        csv_footshock_kwargs.append(session_to_nwb_kwargs)
print(f"{len(csv_footshock_kwargs)} footshock probes converted from CSV")
for session_to_nwb_kwargs in csv_footshock_kwargs:
    session_to_nwb(**session_to_nwb_kwargs)

13 footshock probes converted from CSV


In [3]:
def read_footshock_times(file_path: Path) -> np.ndarray:
    with h5py.File(file_path, "r") as file:
        if "processing/behavior/footshock_times" not in file:
            return np.array([])
        return file["processing/behavior/footshock_times/timestamps"][()]


rows = []
for new_file_path in sorted(output_dir_path.glob("*.nwb")):
    subject_part, session_part = new_file_path.stem.split("_ses-")
    subject_id = subject_part.removeprefix("sub-")
    session_stamp = session_part[-19:]
    subject_folder_path = published_dir_path / f"sub-{subject_id.replace('.', '-')}"
    (published_unknown_file_path,) = subject_folder_path.glob(f"*-Unknown-{session_stamp}*.nwb")
    (published_named_file_path,) = [
        file_path
        for file_path in subject_folder_path.glob(f"*{session_stamp}*.nwb")
        if file_path != published_unknown_file_path
    ]
    new_footshock_times = read_footshock_times(new_file_path)
    unknown_footshock_times = read_footshock_times(published_unknown_file_path)
    rows.append(
        dict(
            subject_id=subject_id,
            start=session_stamp,
            published_named=len(read_footshock_times(published_named_file_path)),
            published_unknown=len(unknown_footshock_times),
            fixed=len(new_footshock_times),
            fixed_equals_unknown=np.array_equal(new_footshock_times, unknown_footshock_times),
        )
    )
pd.DataFrame(rows)

,subject_id,start,published_named,published_unknown,fixed,fixed_equals_unknown
0,276.405,2020-09-11T12-36-24,0,12,12,True
1,276.405,2020-09-29T11-36-53,0,17,17,True
2,299.405,2020-09-10T12-48-26,0,13,13,True
3,299.405,2020-09-29T11-36-53,0,17,17,True
4,300.405,2020-09-10T12-48-54,0,10,10,True
5,300.405,2020-09-29T11-36-53,0,14,14,True
6,307.399,2020-08-26T11-44-48,0,11,11,True
7,307.399,2020-09-11T11-28-25,0,14,14,True
8,308.399,2020-08-26T11-44-48,0,5,5,True
9,308.399,2020-09-11T11-28-25,0,6,6,True


Every reconverted CSV session now has the same footshock times as the MED-PC copy of the run. Thirteen other published footshock probes have no `footshock_times`, but those come from MED-PC files whose shock array `H` is empty; most are runs that ended within minutes.